In [ ]:
!pip install -q torch nvidia-cutlass-dsl triton

# Control Divergence in GPU Kernels

**Control divergence** (or warp divergence) occurs when threads within the same warp (group of 32 threads) take different execution paths due to conditional branching. Since all threads in a warp execute in lockstep (SIMT model), divergent branches must be serialized — the warp first executes one path while masking off inactive threads, then executes the other path.

This means:
- **With divergence**: Both branches execute sequentially, and threads idle while waiting for their branch to run. Effective throughput is halved (or worse).
- **Without divergence**: All threads follow the same path, so no serialization penalty occurs.

In this notebook, we write two CuTe DSL kernels:
1. A **divergent** kernel where even/odd threads within a warp take different code paths
2. A **non-divergent** kernel that computes the same result but without branching

We then benchmark both using `triton.testing.do_bench` to measure the performance impact.

In [ ]:
import torch
import cutlass
import cutlass.cute as cute
import triton

## Kernel With Control Divergence

This kernel branches on `thread_idx % 2`. Within every warp, half the threads (even) take the `if` path and half (odd) take the `else` path. The warp must serialize both branches — first executing the even-thread path, then the odd-thread path (or vice versa).

To make the divergence impact more visible, each branch performs a non-trivial amount of arithmetic (a loop of floating-point operations). This amplifies the cost of serialization.

In [ ]:
BLOCK_SIZE = 256
NUM_ITERATIONS = 100  # loop iterations inside each branch to amplify divergence cost


@cute.kernel
def divergent_kernel(
    gInput: cute.Tensor,
    gOutput: cute.Tensor,
    N: cutlass.Int32,
):
    tid, _, _ = cute.arch.thread_idx()
    bid, _, _ = cute.arch.block_idx()
    idx = bid * BLOCK_SIZE + tid

    if idx < N:
        val = gInput[idx]

        # Control divergence: even and odd threads within the same warp
        # take different paths, forcing serial execution of both branches.
        if tid % 2 == 0:
            for i in range(NUM_ITERATIONS):
                val = val * 1.01 + 0.01
        else:
            for i in range(NUM_ITERATIONS):
                val = val * 0.99 + 0.02

        gOutput[idx] = val

## Kernel Without Control Divergence

This kernel performs the **same total amount of work** as the divergent kernel but avoids branching. Instead of using `if/else`, it uses arithmetic blending: a `selector` (0 or 1) computed from `tid % 2` determines the effective constants via multiplication, so all threads execute the same instruction sequence. No warp serialization occurs.

In [ ]:
@cute.kernel
def non_divergent_kernel(
    gInput: cute.Tensor,
    gOutput: cute.Tensor,
    N: cutlass.Int32,
):
    tid, _, _ = cute.arch.thread_idx()
    bid, _, _ = cute.arch.block_idx()
    idx = bid * BLOCK_SIZE + tid

    if idx < N:
        val = gInput[idx]

        # Branchless: use arithmetic to select constants.
        # selector = 0.0 for even threads, 1.0 for odd threads
        selector = cutlass.Float32(tid % 2)

        # Blend between the two sets of constants:
        #   even: mul_factor=1.01, add_factor=0.01
        #   odd:  mul_factor=0.99, add_factor=0.02
        mul_factor = 1.01 + selector * (0.99 - 1.01)   # 1.01 or 0.99
        add_factor = 0.01 + selector * (0.02 - 0.01)    # 0.01 or 0.02

        for i in range(NUM_ITERATIONS):
            val = val * mul_factor + add_factor

        gOutput[idx] = val

## Host Launch Functions

We wrap each kernel in a `@cute.jit` host function that sets up the grid/block dimensions and launches the kernel. PyTorch tensors passed to `@cute.jit` are implicitly converted to CuTe tensors via the DLPack protocol.

In [ ]:
@cute.jit
def launch_divergent(input_tensor: cute.Tensor, output_tensor: cute.Tensor, N: cutlass.Int32):
    grid = ((N + BLOCK_SIZE - 1) // BLOCK_SIZE, 1, 1)
    block = (BLOCK_SIZE, 1, 1)
    divergent_kernel(input_tensor, output_tensor, N).launch(grid=grid, block=block)


@cute.jit
def launch_non_divergent(input_tensor: cute.Tensor, output_tensor: cute.Tensor, N: cutlass.Int32):
    grid = ((N + BLOCK_SIZE - 1) // BLOCK_SIZE, 1, 1)
    block = (BLOCK_SIZE, 1, 1)
    non_divergent_kernel(input_tensor, output_tensor, N).launch(grid=grid, block=block)

## Correctness Check

Before benchmarking, verify that both kernels produce the same output.

In [7]:
N = 1 << 20  # ~1M elements

input_tensor = torch.randn(N, device="cuda", dtype=torch.float32)
output_divergent = torch.empty_like(input_tensor)
output_non_divergent = torch.empty_like(input_tensor)

launch_divergent(input_tensor, output_divergent, N)
launch_non_divergent(input_tensor, output_non_divergent, N)

print(f"Outputs match: {torch.allclose(output_divergent, output_non_divergent, atol=1e-4)}")
print(f"Max difference: {(output_divergent - output_non_divergent).abs().max().item():.6e}")

AssertionError: Torch not compiled with CUDA enabled

## Benchmarking with `triton.testing.do_bench`

We use Triton's `do_bench` utility to get reliable GPU kernel timings. It handles warmup, synchronization, and returns percentile-based statistics.

We benchmark across multiple input sizes to see how the divergence penalty scales.

In [ ]:
sizes = [2**i for i in range(16, 25)]  # 64K to 16M elements

print(f"{'N':>12} | {'Divergent (ms)':>15} | {'Non-Divergent (ms)':>19} | {'Speedup':>8}")
print("-" * 65)

for N in sizes:
    x = torch.randn(N, device="cuda", dtype=torch.float32)
    out = torch.empty_like(x)

    ms_div = triton.testing.do_bench(lambda: launch_divergent(x, out, N), warmup=25, rep=100)
    ms_nodiv = triton.testing.do_bench(lambda: launch_non_divergent(x, out, N), warmup=25, rep=100)

    speedup = ms_div / ms_nodiv
    print(f"{N:>12,} | {ms_div:>15.4f} | {ms_nodiv:>19.4f} | {speedup:>7.2f}x")

## Analysis

**Why the divergent kernel is slower:**

In the divergent kernel, every warp hits the `if tid % 2 == 0` branch. Since even and odd threads are interleaved within each warp, the warp must:
1. Execute the `if` branch (100 iterations of `val * 1.01 + 0.01`) with odd threads masked off
2. Execute the `else` branch (100 iterations of `val * 0.99 + 0.02`) with even threads masked off

The warp effectively does **2x the serial work** — both loop bodies run, but only half the threads are active during each.

**Why the non-divergent kernel is faster:**

All threads execute the same instruction stream (`val * mul_factor + add_factor`) with different data values. The warp runs one unified loop of 100 iterations with all 32 threads active simultaneously. No serialization occurs.

**Key takeaway:** Avoid branching patterns that split threads within the same warp. When you must branch, try to make the branch condition uniform across the warp (e.g., branch on `warp_id` or `block_idx` rather than `thread_idx % 2`). Alternatively, replace branches with arithmetic blending as shown in the non-divergent kernel.